### Import Dependencies

In [16]:
from pydantic import BaseModel, Field
import os

from qdrant_client import QdrantClient
from qdrant_client.models import Prefetch, Filter, FieldCondition, MatchText, FusionQuery, Document


from langsmith import traceable, get_current_run_tree, Client

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.types import Send, Command

from langchain_core.messages import AIMessage, ToolMessage, convert_to_openai_messages

from jinja2 import Template
from typing import Literal, Dict, Any, Annotated, List, Optional, Sequence
from IPython.display import Image, display
from operator import add
from openai import OpenAI

import openai

import random
import ast
import inspect
import instructor
import json

from utils.utils import get_tool_descriptions, format_ai_message
from utils.tools import add_to_shopping_cart, get_shopping_cart, remove_from_cart, get_formatted_items_context, get_formatted_reviews_context, check_warehouse_availability, reserve_warehouse_items

from langgraph.checkpoint.postgres import PostgresSaver

## Coordinator Agent

In [2]:
class Delegation(BaseModel):
    agent: str
    task: str

class ToolCall(BaseModel):
    name: str
    arguments: dict

class RAGUsedContext(BaseModel):
    id: str = Field(description="The ID of the item used to answer the question")
    description: str = Field(description="Short description of the item used to answer the question")

class CoordinatorAgentResponse(BaseModel):
    next_agent: str
    plan: List[Delegation]
    final_answer: bool = False
    answer: str = ""

In [3]:
class AgentProperties(BaseModel):    
    iteration: int = 0
    final_answer: bool = False
    available_tools: List[Dict[str, Any]] = []
    tool_calls: List[ToolCall] = []


class CoordinatorAgentProperties(BaseModel):    
    iteration: int = 0
    final_answer: bool = False
    plan: List[Delegation] = []
    next_agent: str = ""


class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    user_intent: str = ""
    product_qa_agent: AgentProperties = Field(default_factory=AgentProperties)
    shopping_cart_agent: AgentProperties = Field(default_factory=AgentProperties)
    warehouse_manager_agent: AgentProperties = Field(default_factory=AgentProperties)
    coordinator_agent: CoordinatorAgentProperties = Field(default_factory=CoordinatorAgentProperties)
    answer: str = ""
    references: Annotated[List[RAGUsedContext], add] = []
    user_id: str = ""
    cart_id: str = ""

In [5]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={"ls_provider": "openai", "ls_model_name": "gpt-4.1"}
)
def coordinator_agent(state):

   prompt_template =  """You are a Coordinator Agent as part of a shopping assistant.

Your role is to create plans for solving user queries and delegate the tasks accordingly.
You will be given a conversation history, your task is to create a plan for solving the user's query.
After the plan is created, you should output the next agent to invoke and the task to be performed by that agent.
Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and revise the plan.
If there is a sequence of tasks to be performed by a single agent, you should combine them into a single task.

The possible agents are:

- product_qa_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.

CRITICAL RULES:
- If next_agent is "", final_answer MUST be false
(You cannot delegate the task to an agent and return to the user in the same response)
- If final_answer is true, next_agent MUST be ""
(You must wait for agent results before returning to user)
- If you need to call other agents before answering, set:
next_agent="...", final_answer=false
- After receiving agent results, you can then set:
next_agent="", final_answer=true
- One of the following has to be true:
next_agent is "" and final_answer is true
next_agent is not "" and final_answer is false

Additional instructions:

- Do not route to any agent if the user's query needs clarification. Do it yourself.
- Write the plan to the plan field.
- Write the next agent to invoke to the next_agent field.
- Once you have all the information needed to answer the user's query, you should set the final_answer field to True and output the answer to the user's query.
- The final answer to the user query should be a comprehensive answer that explains the actions that were performed to answer the query.
- Never set final_answer to true if the plan is not complete.
- You should output the next_agent field as well as the plan field.
"""

   template = Template(prompt_template)
   
   prompt = template.render()

   messages = state.messages

   conversation = []

   for message in messages:
        conversation.append(convert_to_openai_messages(message))

   client = instructor.from_openai(OpenAI())

   response, raw_response = client.chat.completions.create_with_completion(
        model="gpt-4.1",
        response_model=CoordinatorAgentResponse,
        messages=[{"role": "system", "content": prompt}, *conversation],
        temperature=0.5,
   )

   if response.final_answer:
      ai_message = [AIMessage(
         content=response.answer,
      )]
   else:
      ai_message = []

   return {
      "messages": ai_message,
      "answer": response.answer,
      "coordinator_agent": {
         "iteration": state.coordinator_agent.iteration + 1,
         "final_answer": response.final_answer,
         "next_agent": response.next_agent,
         "plan": [data.model_dump() for data in response.plan]
      }
   }

In [6]:
initial_state = State(
    messages = [{"role": "user", "content": "what is the wheather now?"}]
)

In [8]:
answer = coordinator_agent(initial_state)

In [9]:
answer

{'messages': [AIMessage(content="I'm here to assist with shopping-related questions, such as finding products, checking availability, or helping with your shopping cart. If you have a question about a product or need help shopping, please let me know!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'answer': "I'm here to assist with shopping-related questions, such as finding products, checking availability, or helping with your shopping cart. If you have a question about a product or need help shopping, please let me know!",
 'coordinator_agent': {'iteration': 1,
  'final_answer': True,
  'next_agent': '',
  'plan': [{'agent': 'product_qa_agent',
    'task': "The user is asking about the weather, which is unrelated to shopping or available products. Clarify the user's intent or redirect to relevant shopping topics."}]}}

In [12]:
initial_state = State(
    messages = [{"role": "user", "content": "Suggest me some earphones?"}]
)

In [13]:
answer_qa = coordinator_agent(initial_state)

In [14]:
answer_qa

{'messages': [],
 'answer': '',
 'coordinator_agent': {'iteration': 1,
  'final_answer': False,
  'next_agent': 'product_qa_agent',
  'plan': [{'agent': 'product_qa_agent',
    'task': 'Suggest a selection of earphones to the user, including a variety of popular or highly rated options.'}]}}

In [15]:
coordinator_eval_dataset = [
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "What is the weather today?"}
            ]
        },
        "outputs": {
            "next_agent": "",
            "coordinator_final_answer": True
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "Can I get some earphones?"}
            ]
        },
        "outputs": {
            "next_agent": "product_qa_agent",
            "coordinator_final_answer": False
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "Can you add an item with ID B09NLTDHQ6 to my cart?"}
            ]
        },
        "outputs": {
            "next_agent": "shopping_cart_agent",
            "coordinator_final_answer": False
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "Can you add those earphones to my cart?"}
            ]
        },
        "outputs": {
            "next_agent": "",
            "coordinator_final_answer": True
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "Can you add the best items to my cart? I am looking for laptop bags."}
            ]
        },
        "outputs": {
            "next_agent": "product_qa_agent",
            "coordinator_final_answer": False
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "Can you find some good reviews for items in my cart?"}
            ]
        },
        "outputs": {
            "next_agent": "shopping_cart_agent",
            "coordinator_final_answer": False
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "Can you put the items with the most positive user reviews to my cart?"}
            ]
        },
        "outputs": {
            "next_agent": "product_qa_agent",
            "coordinator_final_answer": False
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "What kind of stuff do you sell?"}
            ]
        },
        "outputs": {
            "next_agent": "",
            "coordinator_final_answer": True
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "Can you help me with my order?"}
            ]
        },
        "outputs": {
            "next_agent": "",
            "coordinator_final_answer": True
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "Can you add two, ideally red tablets to my cart?"}
            ]
        },
        "outputs": {
            "next_agent": "product_qa_agent",
            "coordinator_final_answer": False
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "Can you reserve a red leather laptop bag for me?"}
            ]
        },
        "outputs": {
            "next_agent": "product_qa_agent",
            "coordinator_final_answer": False
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "Can you reserve my shopping cart?"}
            ]
        },
        "outputs": {
            "next_agent": "shopping_cart_agent",
            "coordinator_final_answer": False
        }
    }
]


### Upload dataset to Langsmith

In [22]:
ls_client = Client(api_key=os.environ["LANGSMITH_API_KEY"])

dataset_name = "coordinator_agent_eval_dataset"

dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="Coordinator Agent evaluation dataset"
)

In [23]:
for item in coordinator_eval_dataset:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={"messages": item["inputs"]["messages"]},
        outputs={
            "next_agent": item["outputs"]["next_agent"],
            "coordinator_final_answer": item["outputs"]["coordinator_final_answer"]
        }
    )

In [24]:
def coordinator_agent_evaluator(run, example):
    next_agent_match = run.outputs["coordinator_agent"]["next_agent"] == example.outputs["next_agent"]
    final_answer_match = run.outputs["coordinator_agent"]["final_answer"] == example.outputs["coordinator_final_answer"]

    return next_agent_match and final_answer_match

In [28]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(State(messages=x["messages"])),
    data= "coordinator_agent_eval_dataset",
    evaluators=[
        coordinator_agent_evaluator
    ],
    experiment_prefix="coordinator_agent_eval_dataset"
)

View the evaluation results for experiment: 'coordinator_agent_eval_dataset-c22ae4bc' at:
https://eu.smith.langchain.com/o/4f3c49ad-7aa9-46c0-bf18-295c3073eb95/datasets/353e4126-212d-4ca9-84b2-7b6b07c19e9a/compare?selectedSessions=11c50a29-4642-4268-8460-df6bb4614667




12it [00:20,  1.69s/it]
